# 실습 08 · 딥러닝 비전 검사 (YOLO)
### 정제·바이알 이미지에서 결함을 자동 검출

**왜 중요한가 (현장 맥락)**
제약 포장/충전 라인에서 **육안검사**는 느리고 사람마다 기준이 다릅니다.
딥러닝 **객체탐지(YOLO)** 는 이미지에서 정제/바이알을 찾고,
깨짐·이물·색상이상·라벨오류를 **실시간·일관되게** 검출합니다.
이것이 AI 비전검사(automated visual inspection)의 핵심입니다.

**이 노트북의 흐름**
1. 사전학습 YOLO 로 **즉시** 물체 탐지 체험 (설치 후 바로 결과)
2. 제약 결함검사에 어떻게 **커스텀 학습**하는지 (데이터 준비법, 코드 골격)
3. 분류형 검사(정상/불량)와의 차이 및 선택 기준

> GPU 런타임 권장: 상단 [런타임]>[런타임 유형 변경]>**T4 GPU** 선택


In [ ]:
# Ultralytics YOLO 설치 (YOLOv8/v11) — 현장 표준 라이브러리
!pip install ultralytics -q
from ultralytics import YOLO
import matplotlib.pyplot as plt
from PIL import Image
print("YOLO 준비 완료 ✅")

## 1. 사전학습 YOLO 로 즉시 탐지 체험
COCO 데이터로 학습된 모델을 불러와, 임의 이미지에서 물체를 탐지합니다.
(제약 결함 전용은 아니지만, **YOLO 사용법**을 30초 만에 체득)


In [ ]:
# 사전학습 모델 로드 (n=nano, 가장 가벼움)
model = YOLO("yolov8n.pt")

# 샘플 이미지에서 탐지 실행
results = model("https://ultralytics.com/images/bus.jpg")
results[0].save(filename="det.jpg")     # 박스 그려서 저장

plt.figure(figsize=(6,8))
plt.imshow(Image.open("det.jpg")); plt.axis("off")
plt.title("YOLO 객체탐지 결과 (박스+클래스+신뢰도)"); plt.show()

# 탐지 결과를 표로
for b in results[0].boxes:
    cls = model.names[int(b.cls)]
    print(f"{cls:10s}  신뢰도={float(b.conf):.2f}")

## 2. ⭐ 제약 결함검사용 커스텀 학습 (개념 + 코드 골격)
실제 현장에서는 **우리 제품 이미지로 직접 학습**합니다. 준비 과정:

**(1) 데이터 수집·라벨링**
- 정상/결함 제품 사진 수백~수천 장
- 라벨링 도구(Roboflow, LabelImg, CVAT)로 결함 위치에 박스 + 클래스 지정
  - 예: `crack`(깨짐), `contamination`(이물), `discolor`(변색), `label_error`

**(2) YOLO 데이터셋 형식** (`data.yaml`)
```yaml
train: images/train
val:   images/val
names:
  0: crack
  1: contamination
  2: discolor
```

**(3) 학습 코드** (실제로는 아래 한 셀이면 끝)


In [ ]:
# ▼ 커스텀 학습 코드 골격 (데이터 준비 후 주석 해제하여 실행)
# model = YOLO("yolov8n.pt")             # 사전학습 가중치에서 시작(전이학습)
# model.train(
#     data="data.yaml",                  # 위에서 만든 데이터 정의
#     epochs=50, imgsz=640, batch=16,
#     patience=10,                       # 조기종료
# )
# # 학습 후 검증 이미지로 성능 확인
# metrics = model.val()                  # mAP, precision, recall 자동 계산
# print(metrics.box.map)                 # mAP@0.5:0.95

print("※ 위는 골격입니다. Roboflow 공개 데이터셋으로 실제 학습을 체험하려면 다음 셀 참고")

### 실제 공개 데이터로 학습 체험하기 (선택)
Roboflow 에는 알약/약품 검출 공개 데이터셋이 많습니다.
Roboflow 계정(무료)에서 데이터셋을 받아 아래처럼 학습하면 됩니다.
```python
!pip install roboflow -q
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_KEY")
dataset = rf.workspace("...").project("pill-detection").version(1).download("yolov8")
model = YOLO("yolov8n.pt")
model.train(data=dataset.location + "/data.yaml", epochs=30, imgsz=640)
```


## 3. 객체탐지 vs 분류 — 언제 무엇을?
| 방식 | 출력 | 제약 검사 예시 | 데이터 라벨링 |
|---|---|---|---|
| **분류(Classification)** | 이미지 전체 정상/불량 | "이 정제 사진 = 불량" | 쉬움(폴더로 구분) |
| **객체탐지(YOLO)** | 결함 **위치+종류** | "왼쪽 위에 crack" | 박스 라벨 필요 |
| **분할(Segmentation)** | 픽셀 단위 영역 | 이물 면적 정량화 | 가장 정밀·비쌈 |

**선택 기준**: 위치가 필요 없고 합격/불합격만 판정하면 **분류**로 충분(빠르고 데이터 적음).
결함 위치·개수·크기를 알아야 하면 **탐지/분할**.


In [ ]:
# 참고: 간단한 정상/불량 '분류'는 몇 줄이면 됩니다 (전이학습)
# 폴더 구조: dataset/train/정상, dataset/train/불량 ...
# model = YOLO("yolov8n-cls.pt")           # 분류 전용 모델
# model.train(data="dataset", epochs=20, imgsz=224)
print("분류/탐지/분할을 목적에 맞게 선택하는 것이 실무의 핵심")

## 정리 & 현장 응용
- YOLO 로 **몇 줄 만에** 객체탐지 실행, 전이학습으로 **우리 제품에 맞춤 학습**
- 라벨링(Roboflow/CVAT) → `data.yaml` → `model.train()` → `model.val()` 파이프라인
- 목적에 따라 **분류 / 탐지 / 분할** 선택
- 제약 적용: 정제·캡슐 외관, 바이알 충전량/이물, 라벨·각인 검사, 시린지 결함
